# MecaniQA - OAT 3: ajuste dos modelos

Equipe Macapá — 07/10/2026

Vamos comparar Random Forest e Gradient Boosting e ajustar o que tiver menor RMSE na validação. O teste final fica separado até a escolha dos parâmetros.

## Executar no Colab

Abra este notebook no Colab e execute todas as células. Quando solicitado, envie somente `mecaniqa_dataset.xlsx`. O código de treinamento está incluído no notebook; não é necessário enviar `pipeline.py`. No VS Code, a planilha pode ficar na pasta `datasets`.


## Organização do experimento

- Alvo: trocas de óleo de amanhã, usando os dados disponíveis até hoje.
- Corte cronológico: 80% dos exemplos válidos para treino e 20% para teste.
- Validação: `TimeSeriesSplit(n_splits=5, gap=1)` dentro do treino.
- Random Forest inicial: 200 árvores, como no pipeline anterior, e `random_state=42`.
- Gradient Boosting inicial: parâmetros padrão e `random_state=42`.
- Grid do Random Forest: 50, 100 e 200 árvores; profundidades 3, 5 e 10, conforme a decisão da equipe.
- Grid do Gradient Boosting: 100, 200 e 300 árvores; profundidades 2 e 3; taxas de aprendizagem 0,05 e 0,1.

A mediana e a escala são aprendidas somente no treino de cada fold. Quantidades negativas são tratadas como ausentes. Os modelos usam as mesmas entradas (`Trocas_Oleo` e `Manutencao_Motor`) para isolar o efeito do ajuste.


In [1]:
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import GridSearchCV, ParameterGrid, TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


ARQUIVO_DADOS = Path("datasets/mecaniqa_dataset.xlsx")
if not ARQUIVO_DADOS.exists():
    ARQUIVO_DADOS = Path("mecaniqa_dataset.xlsx")
if not ARQUIVO_DADOS.exists():
    try:
        # Esta biblioteca só está disponível no ambiente do Colab.
        from google.colab import files  # pyright: ignore[reportMissingImports]
    except ImportError as erro:
        raise FileNotFoundError("Coloque mecaniqa_dataset.xlsx na pasta datasets ou ao lado do notebook.") from erro
    print("Selecione o arquivo mecaniqa_dataset.xlsx do seu computador.")
    files.upload()
    if not ARQUIVO_DADOS.exists():
        raise FileNotFoundError("Envie a planilha com o nome mecaniqa_dataset.xlsx e execute novamente.")


def carregar_dados():
    df = pd.read_excel(ARQUIVO_DADOS)
    df["Data"] = pd.to_datetime(df["Data"], errors="raise")
    df = df.sort_values("Data").set_index("Data")

    if df.index.hasnans or df.index.has_duplicates:
        raise ValueError("A planilha precisa ter datas válidas e sem duplicatas.")
    if not df.index.to_series().diff().dropna().eq(pd.Timedelta(days=1)).all():
        raise ValueError("A planilha precisa ter uma linha por dia, sem intervalos.")

    # Quantidades negativas são entradas inválidas; a imputação ocorre no pipeline.
    colunas = ["Trocas_Oleo", "Manutencao_Motor"]
    df[colunas] = df[colunas].mask(df[colunas] < 0)
    dados = df[colunas].copy()
    dados["Demanda_Amanha"] = df["Trocas_Oleo"].shift(-1)

    # Previsões de amanhã usando somente informações disponíveis até hoje.
    dados["Naive"] = df["Trocas_Oleo"].ffill()
    dados["Media_Movel_7"] = df["Trocas_Oleo"].ffill().rolling(7, min_periods=1).mean()
    dados = dados.dropna(subset=["Demanda_Amanha"])
    return dados, colunas


def criar_pipeline(modelo):
    return Pipeline([
        ("imputacao", SimpleImputer(strategy="median")),
        ("padronizacao", StandardScaler()),
        ("modelo", modelo),
    ])


def calcular_metricas(y_real, previsao):
    real = np.asarray(y_real, dtype=float)
    previsto = np.asarray(previsao, dtype=float)
    positivos = real > 0
    mape = (
        np.mean(np.abs((real[positivos] - previsto[positivos]) / real[positivos])) * 100
        if positivos.any() else np.nan
    )
    return {
        "MAE": mean_absolute_error(real, previsto),
        "RMSE": np.sqrt(mean_squared_error(real, previsto)),
        "MAPE (demanda > 0) %": mape,
    }


def main():
    dados, colunas = carregar_dados()
    corte = int(len(dados) * 0.8)
    treino, teste = dados.iloc[:corte], dados.iloc[corte:]
    X_train, y_train = treino[colunas], treino["Demanda_Amanha"]
    X_test, y_test = teste[colunas], teste["Demanda_Amanha"]

    print(f"Exemplos com alvo válido: {len(dados)}")
    print(f"Treino: {len(treino)} | Teste: {len(teste)}")
    print(f"Datas de referência do treino: {treino.index.min().date()} a {treino.index.max().date()}")
    print(f"Datas de referência do teste: {teste.index.min().date()} a {teste.index.max().date()}")

    # O intervalo protege a fronteira dos folds, pois o alvo é do dia seguinte.
    tscv = TimeSeriesSplit(n_splits=5, gap=1)
    modelos = {
        "Random Forest": criar_pipeline(RandomForestRegressor(n_estimators=200, random_state=42)),
        "Gradient Boosting": criar_pipeline(GradientBoostingRegressor(random_state=42)),
    }
    grids = {
        "Random Forest": {
            "modelo__n_estimators": [50, 100, 200],
            "modelo__max_depth": [3, 5, 10],
        },
        "Gradient Boosting": {
            "modelo__n_estimators": [100, 200, 300],
            "modelo__max_depth": [2, 3],
            "modelo__learning_rate": [0.05, 0.1],
        },
    }

    inicio = perf_counter()
    resultados_cv = {}
    for nome, modelo in modelos.items():
        scores = cross_val_score(
            modelo, X_train, y_train, cv=tscv,
            scoring="neg_root_mean_squared_error", n_jobs=1, error_score="raise",
        )
        resultados_cv[nome] = -scores.mean()
        print(f"{nome} inicial - RMSE médio na validação: {resultados_cv[nome]:.4f}", flush=True)

    # A escolha usa apenas a validação dentro dos 80% de treino.
    vencedor = min(resultados_cv, key=resultados_cv.get)
    grid = grids[vencedor]
    print(f"\nModelo escolhido: {vencedor}")
    print(f"Combinações: {len(ParameterGrid(grid))} | Folds: {tscv.n_splits}", flush=True)
    busca = GridSearchCV(
        modelos[vencedor], param_grid=grid, cv=tscv,
        scoring="neg_root_mean_squared_error", n_jobs=1,
        refit=True, error_score="raise",
    )
    busca.fit(X_train, y_train)
    print("best_params_:", busca.best_params_)
    print(f"RMSE médio na validação após ajuste: {-busca.best_score_:.4f}")

    # O teste final é usado somente depois da escolha e do ajuste.
    previsoes = pd.DataFrame(index=y_test.index + pd.Timedelta(days=1))
    previsoes.index.name = "Data_da_previsao"
    previsoes["Real"] = y_test.to_numpy()
    previsoes["Naive"] = teste["Naive"].to_numpy()
    previsoes["Média móvel 7 dias"] = teste["Media_Movel_7"].to_numpy()
    for nome, modelo in modelos.items():
        modelo.fit(X_train, y_train)
        previsoes[f"{nome} inicial"] = modelo.predict(X_test)
    nome_ajustado = f"{vencedor} ajustado"
    previsoes[nome_ajustado] = busca.predict(X_test)

    metricas = pd.DataFrame({
        nome: calcular_metricas(previsoes["Real"], previsoes[nome])
        for nome in previsoes.columns if nome != "Real"
    }).T
    metricas.index.name = "Modelo"
    print("\nMétricas no mesmo teste final:")
    print(metricas.round(4).to_string())
    print(f"MAPE calculado em {int((y_test > 0).sum())} de {len(y_test)} dias; demanda zero não tem MAPE definido.")

    rmse_inicial = metricas.loc[f"{vencedor} inicial", "RMSE"]
    rmse_ajustado = metricas.loc[nome_ajustado, "RMSE"]
    reducao = (rmse_inicial - rmse_ajustado) / rmse_inicial * 100 if rmse_inicial else np.nan
    print(f"\nRedução percentual do RMSE no teste: {reducao:.2f}%")
    if not np.isfinite(reducao):
        print("O erro inicial é zero; a redução percentual não é definida.")
    elif reducao < 0:
        print("O ajuste aumentou o erro no teste. A melhora na validação não se confirmou.")
    elif reducao == 0:
        print("O ajuste manteve o erro do modelo inicial.")
    print(f"Tempo total: {perf_counter() - inicio:.2f} segundos")

    return {
        "metricas": metricas,
        "previsoes": previsoes,
        "busca": busca,
        "rmse_cv_inicial": resultados_cv,
        "modelo_escolhido": vencedor,
        "reducao_rmse_percentual": reducao,
    }



resultado = main()


Exemplos com alvo válido: 724
Treino: 579 | Teste: 145
Datas de referência do treino: 2024-01-01 a 2025-08-07
Datas de referência do teste: 2025-08-08 a 2025-12-30


Random Forest inicial - RMSE médio na validação: 11.9766


Gradient Boosting inicial - RMSE médio na validação: 12.4709



Modelo escolhido: Random Forest
Combinações: 9 | Folds: 5


best_params_: {'modelo__max_depth': 3, 'modelo__n_estimators': 50}
RMSE médio na validação após ajuste: 10.1835



Métricas no mesmo teste final:
                              MAE    RMSE  MAPE (demanda > 0) %
Modelo                                                         
Naive                      6.4207  8.8517               26.6387
Média móvel 7 dias         7.3478  8.1193               30.8942
Random Forest inicial      7.0181  8.7263               29.7019
Gradient Boosting inicial  6.2699  7.8992               26.5733
Random Forest ajustado     6.3321  7.7908               26.8577
MAPE calculado em 145 de 145 dias; demanda zero não tem MAPE definido.

Redução percentual do RMSE no teste: 10.72%
Tempo total: 5.94 segundos


In [2]:
resultado['metricas'].round(4)


,MAE,RMSE,MAPE (demanda > 0) %
Modelo,,,
Naive,6.4207,8.8517,26.6387
Média móvel 7 dias,7.3478,8.1193,30.8942
Random Forest inicial,7.0181,8.7263,29.7019
Gradient Boosting inicial,6.2699,7.8992,26.5733
Random Forest ajustado,6.3321,7.7908,26.8577


In [3]:
import pandas as pd

busca = resultado['busca']
comparacao_grid = pd.DataFrame(busca.cv_results_)
comparacao_grid['RMSE_validacao'] = -comparacao_grid['mean_test_score']
comparacao_grid[['params', 'RMSE_validacao', 'rank_test_score']].sort_values('rank_test_score')


,params,RMSE_validacao,rank_test_score
0,"{'modelo__max_depth': 3, 'modelo__n_estimators...",10.183487,1
2,"{'modelo__max_depth': 3, 'modelo__n_estimators...",10.349628,2
1,"{'modelo__max_depth': 3, 'modelo__n_estimators...",10.357966,3
3,"{'modelo__max_depth': 5, 'modelo__n_estimators...",10.904212,4
5,"{'modelo__max_depth': 5, 'modelo__n_estimators...",11.042202,5
4,"{'modelo__max_depth': 5, 'modelo__n_estimators...",11.059741,6
6,"{'modelo__max_depth': 10, 'modelo__n_estimator...",11.649612,7
8,"{'modelo__max_depth': 10, 'modelo__n_estimator...",11.822365,8
7,"{'modelo__max_depth': 10, 'modelo__n_estimator...",11.842005,9


In [4]:
resultado['previsoes'].head(10)


,Real,Naive,Média móvel 7 dias,Random Forest inicial,Gradient Boosting inicial,Random Forest ajustado
Data_da_previsao,,,,,,
2025-08-09,33.0,35.0,28.285714,42.106927,32.701260,29.766215
2025-08-10,32.0,33.0,27.428571,30.018623,30.330026,29.715721
2025-08-11,22.0,32.0,26.857143,26.526583,28.380826,29.492504
2025-08-12,17.0,22.0,27.285714,20.473774,23.571352,23.176845
2025-08-13,19.0,17.0,26.571429,19.445085,20.530224,21.179248
2025-08-14,17.0,19.0,25.428571,22.046369,21.655231,21.691567
2025-08-15,35.0,17.0,25.000000,19.445085,20.530224,21.179248
2025-08-16,35.0,35.0,25.000000,32.480999,31.215664,30.010306
2025-08-17,37.0,35.0,25.285714,32.480999,31.215664,30.010306


## Leitura dos resultados

A redução percentual é `(RMSE inicial - RMSE ajustado) / RMSE inicial * 100`, usando o mesmo teste. Um resultado negativo significa que o ajuste piorou o erro no teste. A escolha do modelo e dos parâmetros não usa esse teste.

MAE e RMSE incluem todos os dias. O MAPE considera apenas demanda maior que zero e não representa os dias de demanda zero. A quantidade de dias incluídos aparece na saída.

O erro mede a diferença na quantidade de trocas de óleo. Para convertê-lo em prejuízo financeiro, seria necessário informar os custos de falta e de excesso de estoque.
